# 🧹 Kaggle Data Cleaning & Normalization Pipeline

This notebook cleans raw TSV files from the Kaggle dataset and exports the cleaned files as a **downloadable ZIP archive**.

**Input Paths (Kaggle):**
- Train: `/kaggle/input/datasets/cdbharath234/dataset3/student_resource/dataset/train/`
- Test: `/kaggle/input/datasets/cdbharath234/dataset3/student_resource/dataset/test/`

**Output:**
- Cleaned TSVs → `/kaggle/working/cleaned/`
- ZIP Archive  → `/kaggle/working/cleaned_data.zip` (download from Kaggle output panel)


---
## 📦 Cell 1: Clone Repository & Install Dependencies

In [ ]:
import os

# Navigate to Kaggle working directory
%cd /kaggle/working

# Clean old clone if re-running
!rm -rf ml_challenge

# Clone GitHub repository
!git clone https://github.com/BharathPESU/ml_challenge.git

# Navigate to source code directory
%cd /kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution

# Switch to your branch and pull latest code
!git checkout Bharath.2004
!git pull origin Bharath.2004

# Install dependencies
%pip install -r requirements.txt --quiet

print("\n✓ Repository cloned and dependencies installed!")
print("✓ Current directory:", os.getcwd())

---
## ⚙️ Cell 2: Environment & Module Setup

In [ ]:
import os
import sys
import gc
import time
import zipfile
import pandas as pd
from IPython.display import display, FileLink

# Add src directory to Python search path
SRC_DIR = "/kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from cleaning import clean_source_dataframe
from io_utils import read_tsv

print("✓ Imports and cleaning modules initialized successfully!")
print(f"✓ Current working directory: {os.getcwd()}")

---
## 📁 Cell 3: Path & Output Directory Configuration

In [ ]:
# Kaggle dataset input paths
BASE_INPUT_DIR = "/kaggle/input/datasets/cdbharath234/dataset3/student_resource/dataset"
TRAIN_DIR = os.path.join(BASE_INPUT_DIR, "train")
TEST_DIR  = os.path.join(BASE_INPUT_DIR, "test")

# Output directories
CLEANED_DIR = "/kaggle/working/cleaned"
ZIP_PATH    = "/kaggle/working/cleaned_data.zip"

os.makedirs(CLEANED_DIR, exist_ok=True)

print("📁 Path Configuration:")
print(f"   Train Input Dir : {TRAIN_DIR}")
print(f"   Test Input Dir  : {TEST_DIR}")
print(f"   Cleaned Output  : {CLEANED_DIR}")
print(f"   ZIP Output      : {ZIP_PATH}")

# Verify input files exist
sources = {
    "train_source1": os.path.join(TRAIN_DIR, "train_source1.tsv"),
    "train_source2": os.path.join(TRAIN_DIR, "train_source2.tsv"),
    "train_source3": os.path.join(TRAIN_DIR, "train_source3.tsv"),
    "test_source1" : os.path.join(TEST_DIR,  "test_source1.tsv"),
    "test_source2" : os.path.join(TEST_DIR,  "test_source2.tsv"),
    "test_source3" : os.path.join(TEST_DIR,  "test_source3.tsv"),
}

print("\n📋 Input File Status:")
for name, path in sources.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1_048_576 if exists else 0
    status = f"✅ Found ({size_mb:.1f} MB)" if exists else "❌ MISSING"
    print(f"   {name:20s}: {status}")

---
## 🛠️ Cell 4: Memory-Safe Chunked Cleaning Function

In [ ]:
def process_and_clean_file(
    input_path: str,
    output_path: str,
    source_label: str,
    chunksize: int = 500000
) -> str:
    """
    Reads a raw TSV in memory-safe chunks, applies text normalization & cleaning,
    and writes the result incrementally to output_path.
    """
    if not os.path.exists(input_path):
        print(f"⚠️  Input file not found, skipping: {input_path}")
        return output_path

    start_time = time.time()
    print(f"⏳ [{source_label}] Cleaning: {os.path.basename(input_path)} ...")

    total_rows = 0
    first_chunk = True

    for chunk in pd.read_csv(
        input_path, sep="\t", dtype=str,
        keep_default_na=False, chunksize=chunksize
    ):
        cleaned = clean_source_dataframe(chunk, source_label=f"{source_label}_chunk")
        mode   = "w" if first_chunk else "a"
        header = first_chunk
        cleaned.to_csv(output_path, sep="\t", index=False, mode=mode, header=header)
        total_rows  += len(cleaned)
        first_chunk  = False
        del chunk, cleaned
        gc.collect()

    elapsed = time.time() - start_time
    size_mb = os.path.getsize(output_path) / 1_048_576
    print(f"   ✅ {total_rows:,} rows → {os.path.basename(output_path)} ({size_mb:.1f} MB, {elapsed:.1f}s)")
    return output_path

---
## 🏃 Cell 5: Clean Training Data (S1, S2, S3)

In [ ]:
print("=" * 60)
print("  CLEANING TRAINING DATA")
print("=" * 60)

process_and_clean_file(
    os.path.join(TRAIN_DIR, "train_source1.tsv"),
    os.path.join(CLEANED_DIR, "train_source1_clean.tsv"),
    "train_S1"
)

process_and_clean_file(
    os.path.join(TRAIN_DIR, "train_source2.tsv"),
    os.path.join(CLEANED_DIR, "train_source2_clean.tsv"),
    "train_S2"
)

process_and_clean_file(
    os.path.join(TRAIN_DIR, "train_source3.tsv"),
    os.path.join(CLEANED_DIR, "train_source3_clean.tsv"),
    "train_S3"
)

print("\n🎉 All training sources cleaned!")

---
## 🧪 Cell 6: Clean Test Data (S1, S2, S3)

In [ ]:
print("=" * 60)
print("  CLEANING TEST DATA")
print("=" * 60)

process_and_clean_file(
    os.path.join(TEST_DIR, "test_source1.tsv"),
    os.path.join(CLEANED_DIR, "test_source1_clean.tsv"),
    "test_S1"
)

process_and_clean_file(
    os.path.join(TEST_DIR, "test_source2.tsv"),
    os.path.join(CLEANED_DIR, "test_source2_clean.tsv"),
    "test_S2"
)

process_and_clean_file(
    os.path.join(TEST_DIR, "test_source3.tsv"),
    os.path.join(CLEANED_DIR, "test_source3_clean.tsv"),
    "test_S3"
)

print("\n🎉 All test sources cleaned!")

---
## 📋 Cell 7: Inspect Cleaned Output Files

In [ ]:
cleaned_files = sorted([f for f in os.listdir(CLEANED_DIR) if f.endswith(".tsv")])
print(f"📊 Total Cleaned TSV Files: {len(cleaned_files)}\n")

total_size_mb = 0
for fname in cleaned_files:
    fpath = os.path.join(CLEANED_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1_048_576
    total_size_mb += size_mb
    df_peek = pd.read_csv(fpath, sep="\t", nrows=3, dtype=str, keep_default_na=False)
    print(f"📄 {fname}  ({size_mb:.1f} MB)")
    print(f"   Columns: {list(df_peek.columns)}")
    display(df_peek.head(2))
    print("-" * 70)

print(f"\n💾 Total cleaned data size: {total_size_mb:.1f} MB")

---
## 🗜️ Cell 8: Package Cleaned Data into ZIP for Download

In [ ]:
import zipfile
from IPython.display import FileLink

print(f"⏳ Creating ZIP archive: {ZIP_PATH} ...")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fname in sorted(os.listdir(CLEANED_DIR)):
        if fname.endswith(".tsv"):
            fpath = os.path.join(CLEANED_DIR, fname)
            # Store inside a 'cleaned/' subfolder inside the zip
            zf.write(fpath, arcname=os.path.join("cleaned", fname))
            file_mb = os.path.getsize(fpath) / 1_048_576
            print(f"   + cleaned/{fname}  ({file_mb:.1f} MB)")

zip_mb = os.path.getsize(ZIP_PATH) / 1_048_576
print(f"\n✅ ZIP created: {ZIP_PATH}  ({zip_mb:.1f} MB)")
print("   👇 Download it from the Kaggle Output panel on the right.")

---
## ✅ Cell 9: Verify ZIP Contents

In [ ]:
import zipfile

print(f"📦 Contents of {os.path.basename(ZIP_PATH)}:\n")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for info in zf.infolist():
        size_mb = info.file_size / 1_048_576
        compressed_mb = info.compress_size / 1_048_576
        ratio = (1 - info.compress_size / info.file_size) * 100 if info.file_size > 0 else 0
        print(f"   {info.filename:<40s}  {size_mb:6.1f} MB  →  {compressed_mb:5.1f} MB  ({ratio:.0f}% saved)")

zip_mb = os.path.getsize(ZIP_PATH) / 1_048_576
print(f"\n💾 Total ZIP size on disk: {zip_mb:.1f} MB")
print("\n📥 To download:")
print("   1. Go to the Kaggle notebook output panel (right sidebar)")
print("   2. Find 'cleaned_data.zip' and click the download button")
print("   3. Extract it locally into your dataset/cleaned/ folder")